# image-to-world: free SAM2 smoke test on Kaggle

Before running, enable **Settings → Accelerator → GPU** and **Internet**. Then use **Run All**. This notebook downloads the public `image-to-world` repository and Meta's SAM2.1 tiny checkpoint, creates a synthetic image, generates automatic masks, filters them, displays the contact sheet, and produces a ZIP download.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle, enable Settings -> Accelerator -> GPU, then restart and Run All.")

print("GPU:", torch.cuda.get_device_name(0))
install_env = os.environ.copy()
install_env["SAM2_BUILD_CUDA"] = "0"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/sam2.git"],
    check=True,
    env=install_env,
)

repo = Path("/kaggle/working/image-to-world")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DomEscobar/image-to-world.git", str(repo)], check=True)


In [ ]:
import urllib.request

checkpoint = Path("/kaggle/working/sam2.1_hiera_tiny.pt")
checkpoint_url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt"
if not checkpoint.exists():
    print("Downloading SAM2.1 tiny checkpoint...")
    urllib.request.urlretrieve(checkpoint_url, checkpoint)
if checkpoint.stat().st_size < 10_000_000:
    raise RuntimeError("Checkpoint download is unexpectedly small; delete it and retry with Internet enabled.")
print(f"Checkpoint ready: {checkpoint.stat().st_size / 1_000_000:.1f} MB")


In [ ]:
from PIL import Image, ImageDraw

work = Path("/kaggle/working/image-to-world-smoke/work")
work.mkdir(parents=True, exist_ok=True)
source = work / "source.png"

image = Image.new("RGB", (800, 600), (244, 239, 230))
draw = ImageDraw.Draw(image)
draw.polygon([(0, 520), (200, 480), (500, 540), (800, 500), (800, 600), (0, 600)], fill=(85, 150, 90), outline=(30, 80, 35), width=5)
draw.ellipse((380, 300, 470, 400), fill=(65, 125, 230), outline=(20, 55, 130), width=5)
draw.rectangle((250, 380, 400, 410), fill=(180, 110, 65), outline=(90, 45, 20), width=4)
draw.polygon([(600, 300), (620, 260), (640, 300)], fill=(220, 65, 65), outline=(120, 20, 20))
draw.ellipse((120, 200, 150, 230), fill=(245, 195, 45), outline=(130, 90, 5), width=3)
image.save(source)
display(image)


In [ ]:
run_env = os.environ.copy()
run_env.update({
    "SAM2_CHECKPOINT": str(checkpoint),
    "SAM2_CONFIG": "configs/sam2.1/sam2.1_hiera_t.yaml",
    "SAM2_DEVICE": "cuda",
    "SAM2_POINTS_PER_SIDE": "16",
    "SAM2_PRED_IOU": "0.8",
})

masks = work / "masks"
filtered = work / "filtered"
contact = work / "contact.png"

subprocess.run(
    [sys.executable, str(repo / "scripts/segment.py"), str(source), "--out", str(masks), "--backend", "local"],
    check=True,
    env=run_env,
)
subprocess.run(
    [sys.executable, str(repo / "scripts/filter_masks.py"), str(masks), "--out", str(filtered), "--contact", str(contact), "--source", str(source)],
    check=True,
)


In [ ]:
import json
import shutil
from IPython.display import FileLink, display

with (filtered / "manifest.json").open() as handle:
    manifest = json.load(handle)
print(json.dumps(manifest["report"], indent=2))
display(Image.open(contact))

archive_base = Path("/kaggle/working/image-to-world-sam2-results")
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=work.parent, base_dir=work.name))
print("Download the complete result:")
display(FileLink(str(archive)))
